# Track V — V1b corpus pins and selection set S

This notebook records the V1b run governed by D49, D50 and D56. It:

- verifies and audits the MUSAN noise and RIRS_NOISES RIR corpora;
- records corpus, licence and FFmpeg pins;
- creates deterministic 50/50 training/selection corpus inventories with seed 2026;
- extracts only the frozen FoR `val-A` sources;
- builds the five-condition selection set S; and
- validates all 6,650 manifest records without committing audio.

The notebook never accesses DEEP-VOICE. Audio archives, extracted audio, generated waveforms and temporary checkpoints are not repository artifacts.


## 1. Runtime, Drive checkpoint and repository

Use a Python 3.13 Colab runtime with at least 30 GB free. The Drive checkpoint makes the long corpus run recoverable after a runtime reset.


In [ ]:
from pathlib import Path
import subprocess

from google.colab import drive

drive.mount("/content/drive")

REPO = Path("/content/voice-spoofing-detection-v1b")
BRANCH = "feat/stageV1-corpus-pins"

if REPO.exists():
    assert (REPO / ".git").is_dir(), REPO
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            "--single-branch",
            "https://github.com/kahlanzzi/voice-spoofing-detection.git",
            str(REPO),
        ],
        check=True,
    )

subprocess.run(
    ["git", "status", "--short", "--branch"],
    cwd=REPO,
    check=True,
)
subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    check=True,
)


In [ ]:
import hashlib
import json
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy
import pandas
import scipy
import soundfile
import yaml

REPO = Path("/content/voice-spoofing-detection-v1b")
CHECKPOINT_ROOT = Path(
    "/content/drive/MyDrive/voice-spoofing-detection-v1b-checkpoint"
)

WORK_ROOT = Path("/content/v1b_work")
DOWNLOAD_DIR = WORK_ROOT / "downloads"
EXTRACT_DIR = WORK_ROOT / "extracted"
EVIDENCE_DIR = WORK_ROOT / "evidence"

for directory in (
    DOWNLOAD_DIR,
    EXTRACT_DIR,
    EVIDENCE_DIR,
    CHECKPOINT_ROOT,
):
    directory.mkdir(parents=True, exist_ok=True)

assert sys.version_info[:2] == (3, 13), sys.version
assert REPO.is_dir(), REPO

free_bytes = shutil.disk_usage("/content").free
assert free_bytes >= 30 * 1024**3, (
    f"Less than 30 GB free: {free_bytes / 1024**3:.2f} GB"
)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Free /content space:", round(free_bytes / 1024**3, 2), "GB")
print("NumPy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("SciPy:", scipy.__version__)
print("soundfile:", soundfile.__version__)
print("PyYAML:", yaml.__version__)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()

RIRS_URL = (
    "https://openslr.trmal.net/resources/28/rirs_noises.zip"
)
MUSAN_URL = (
    "https://openslr.trmal.net/resources/17/musan.tar.gz"
)

RIRS_ARCHIVE = DOWNLOAD_DIR / "rirs_noises.zip"
MUSAN_ARCHIVE = DOWNLOAD_DIR / "musan.tar.gz"

subprocess.run(
    [
        "wget",
        "--continue",
        "--progress=dot:giga",
        "-O",
        str(RIRS_ARCHIVE),
        RIRS_URL,
    ],
    check=True,
)

subprocess.run(
    [
        "wget",
        "--continue",
        "--progress=dot:giga",
        "-O",
        str(MUSAN_ARCHIVE),
        MUSAN_URL,
    ],
    check=True,
)

expected = {
    "RIRS_NOISES": {
        "path": RIRS_ARCHIVE,
        "bytes": 1_311_166_223,
        "sha256": (
            "3b50cfde915b3984738169b4beb341e9f6b8062ae4c2076146c5db71c2c05dc7"
        ),
        "url": RIRS_URL,
    },
    "MUSAN": {
        "path": MUSAN_ARCHIVE,
        "bytes": 11_086_114_085,
        "sha256": (
            "86d1061c7e15b5c9e906777685c519701df51bfde3001e1070dcc9ffac955ee1"
        ),
        "url": MUSAN_URL,
    },
}

download_record = {
    "schema_version": 1,
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "archives": {},
}

for corpus, item in expected.items():
    actual_bytes = item["path"].stat().st_size
    actual_sha256 = sha256_file(item["path"])

    assert actual_bytes == item["bytes"], (
        corpus,
        actual_bytes,
        item["bytes"],
    )
    assert actual_sha256 == item["sha256"], (
        corpus,
        actual_sha256,
        item["sha256"],
    )

    download_record["archives"][corpus] = {
        "url": item["url"],
        "filename": item["path"].name,
        "bytes": actual_bytes,
        "sha256": actual_sha256,
    }

    print(
        corpus,
        "verified:",
        actual_bytes,
        actual_sha256,
    )

record_path = EVIDENCE_DIR / "archive_downloads.json"

record_path.write_text(
    json.dumps(
        download_record,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)

shutil.copy2(
    record_path,
    CHECKPOINT_ROOT / record_path.name,
)

print("\nArchive downloads and hashes verified")
print("Checkpoint saved:", CHECKPOINT_ROOT / record_path.name)

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
Free /content space: 107.59 GB
NumPy: 2.4.6
pandas: 2.2.3
SciPy: 1.16.3
soundfile: 0.14.0
PyYAML: 6.0.3
RIRS_NOISES verified: 1311166223 3b50cfde915b3984738169b4beb341e9f6b8062ae4c2076146c5db71c2c05dc7
MUSAN verified: 11086114085 86d1061c7e15b5c9e906777685c519701df51bfde3001e1070dcc9ffac955ee1

Archive downloads and hashes verified
Checkpoint saved: /content/drive/MyDrive/voice-spoofing-detection-v1b-checkpoint/archive_downloads.json


## 2. Safe corpus extraction and licence evidence


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import tarfile
import zipfile
from collections import Counter
from pathlib import Path, PurePosixPath

WORK_ROOT = Path("/content/v1b_work")
DOWNLOAD_DIR = WORK_ROOT / "downloads"
EXTRACT_DIR = WORK_ROOT / "extracted"
EVIDENCE_DIR = WORK_ROOT / "evidence"

CHECKPOINT_ROOT = Path(
    "/content/drive/MyDrive/voice-spoofing-detection-v1b-checkpoint"
)
CHECKPOINT_EVIDENCE = CHECKPOINT_ROOT / "evidence"
CHECKPOINT_EVIDENCE.mkdir(parents=True, exist_ok=True)

RIRS_ARCHIVE = DOWNLOAD_DIR / "rirs_noises.zip"
MUSAN_ARCHIVE = DOWNLOAD_DIR / "musan.tar.gz"

RIRS_EXTRACT_ROOT = EXTRACT_DIR / "rirs_noises"
RIRS_ROOT = RIRS_EXTRACT_ROOT / "RIRS_NOISES"
RIRS_FILTERED_ROOT = WORK_ROOT / "filtered" / "rirs_only"

MUSAN_EXTRACT_ROOT = EXTRACT_DIR / "musan"
MUSAN_ROOT = MUSAN_EXTRACT_ROOT / "musan"
MUSAN_NOISE_ROOT = MUSAN_ROOT / "noise"

assert RIRS_ARCHIVE.is_file()
assert MUSAN_ARCHIVE.is_file()
assert not RIRS_EXTRACT_ROOT.exists()
assert not MUSAN_EXTRACT_ROOT.exists()
assert not RIRS_FILTERED_ROOT.exists()

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()

# Validate and extract the official RIRS ZIP.
RIRS_EXTRACT_ROOT.mkdir(parents=True)

with zipfile.ZipFile(RIRS_ARCHIVE) as archive:
    bad_member = archive.testzip()
    assert bad_member is None, bad_member

    for member in archive.infolist():
        member_path = PurePosixPath(member.filename)

        assert not member_path.is_absolute()
        assert ".." not in member_path.parts

    archive.extractall(RIRS_EXTRACT_ROOT)

assert RIRS_ROOT.is_dir()

print("RIRS archive extracted")

# Extract only the MUSAN noise scope.
MUSAN_EXTRACT_ROOT.mkdir(parents=True)
musan_member_count = 0
musan_extracted_count = 0
musan_root_resolved = MUSAN_EXTRACT_ROOT.resolve()

with tarfile.open(MUSAN_ARCHIVE, mode="r:gz") as archive:
    for member in archive:
        musan_member_count += 1
        member_path = PurePosixPath(member.name)

        assert not member_path.is_absolute()
        assert ".." not in member_path.parts

        if not member.name.startswith("musan/noise/"):
            continue

        target = (
            MUSAN_EXTRACT_ROOT / member.name
        ).resolve()

        assert target.is_relative_to(
            musan_root_resolved
        )

        archive.extract(
            member,
            path=MUSAN_EXTRACT_ROOT,
            filter="data",
        )
        musan_extracted_count += 1

assert MUSAN_NOISE_ROOT.is_dir()

print("MUSAN noise scope extracted")

# Identify the exact RIR scope.
selected_rirs = []
excluded_real_area_noise = []

for source in sorted(RIRS_ROOT.rglob("*.wav")):
    relative = source.relative_to(RIRS_ROOT)
    filename = source.name.lower()

    is_simulated_rir = (
        relative.parts[0] == "simulated_rirs"
    )

    is_real_rir = (
        relative.parts[0]
        == "real_rirs_isotropic_noises"
        and (
            "_rir_" in filename
            or filename.startswith("air_")
        )
    )

    is_real_area_noise = (
        relative.parts[0]
        == "real_rirs_isotropic_noises"
        and "_noise_" in filename
    )

    if is_simulated_rir or is_real_rir:
        selected_rirs.append(
            (source, relative)
        )
    elif is_real_area_noise:
        excluded_real_area_noise.append(
            source
        )

simulated_count = sum(
    relative.parts[0] == "simulated_rirs"
    for _, relative in selected_rirs
)
real_count = sum(
    relative.parts[0]
    == "real_rirs_isotropic_noises"
    for _, relative in selected_rirs
)

assert simulated_count == 60_000
assert real_count == 325
assert len(excluded_real_area_noise) == 92
assert len(selected_rirs) == 60_325

for source, relative in selected_rirs:
    destination = RIRS_FILTERED_ROOT / relative
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    os.link(source, destination)

print("Filtered RIR collection created")

# Download the official licence and resource records.
sources = {
    "openslr17.html": "https://www.openslr.org/17/",
    "CC-BY-4.0.txt": (
        "https://creativecommons.org/licenses/by/4.0/legalcode.txt"
    ),
    "openslr28.html": "https://www.openslr.org/28/",
    "Apache-2.0.txt": (
        "https://www.apache.org/licenses/LICENSE-2.0.txt"
    ),
}

for filename, url in sources.items():
    subprocess.run(
        [
            "wget",
            "--quiet",
            "--output-document",
            str(EVIDENCE_DIR / filename),
            url,
        ],
        check=True,
    )

musan_noise_files = sorted(
    path
    for path in MUSAN_NOISE_ROOT.rglob("*")
    if path.is_file()
    and path.suffix.lower()
    in {".wav", ".flac", ".mp3", ".ogg"}
)

musan_categories = Counter(
    path.relative_to(
        MUSAN_NOISE_ROOT
    ).parts[0]
    for path in musan_noise_files
)

assert len(musan_noise_files) == 930
assert musan_categories == {
    "free-sound": 843,
    "sound-bible": 87,
}

musan_metadata = {
    "schema_version": 1,
    "corpus": "MUSAN",
    "scope": "noise_only",
    "open_slr_id": 17,
    "declared_license": "CC BY 4.0",
    "noise_files": 930,
    "categories": dict(
        sorted(musan_categories.items())
    ),
    "archive_sha256": (
        "86d1061c7e15b5c9e906777685c519701df51bfde3001e1070dcc9ffac955ee1"
    ),
    "license_sha256": sha256_file(
        EVIDENCE_DIR / "CC-BY-4.0.txt"
    ),
    "resource_page_sha256": sha256_file(
        EVIDENCE_DIR / "openslr17.html"
    ),
    "archive_member_count": (
        musan_member_count
    ),
    "extracted_member_count": (
        musan_extracted_count
    ),
}

rirs_metadata = {
    "schema_version": 1,
    "corpus": "RIRS_NOISES",
    "scope": "room_impulse_responses_only",
    "open_slr_id": 28,
    "declared_license": "Apache 2.0",
    "rir_files": 60_325,
    "simulated_rirs": simulated_count,
    "real_rirs": real_count,
    "excluded_real_area_noise_files": (
        len(excluded_real_area_noise)
    ),
    "archive_sha256": (
        "3b50cfde915b3984738169b4beb341e9f6b8062ae4c2076146c5db71c2c05dc7"
    ),
    "license_sha256": sha256_file(
        EVIDENCE_DIR / "Apache-2.0.txt"
    ),
    "resource_page_sha256": sha256_file(
        EVIDENCE_DIR / "openslr28.html"
    ),
    "selection_rule": {
        "simulated": "all WAV files under simulated_rirs",
        "real": (
            "filenames containing _rir_ "
            "or beginning with air_"
        ),
        "excluded": (
            "filenames containing _noise_"
        ),
    },
}

for filename, record in (
    ("musan_metadata.json", musan_metadata),
    ("rirs_metadata.json", rirs_metadata),
):
    evidence_path = EVIDENCE_DIR / filename

    evidence_path.write_text(
        json.dumps(
            record,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )

    shutil.copy2(
        evidence_path,
        CHECKPOINT_EVIDENCE / filename,
    )

for filename in sources:
    shutil.copy2(
        EVIDENCE_DIR / filename,
        CHECKPOINT_EVIDENCE / filename,
    )

print("\nMUSAN metadata:")
print(
    json.dumps(
        musan_metadata,
        indent=2,
        sort_keys=True,
    )
)

print("\nRIRS metadata:")
print(
    json.dumps(
        rirs_metadata,
        indent=2,
        sort_keys=True,
    )
)

print(
    "\nPersistent evidence saved:",
    CHECKPOINT_EVIDENCE,
)

RIRS archive extracted
MUSAN noise scope extracted
Filtered RIR collection created

MUSAN metadata:
{
  "archive_member_count": 2050,
  "archive_sha256": "86d1061c7e15b5c9e906777685c519701df51bfde3001e1070dcc9ffac955ee1",
  "categories": {
    "free-sound": 843,
    "sound-bible": 87
  },
  "corpus": "MUSAN",
  "declared_license": "CC BY 4.0",
  "extracted_member_count": 936,
  "license_sha256": "9ba9550ad48438d0836ddab3da480b3b69ffa0aac7b7878b5a0039e7ab429411",
  "noise_files": 930,
  "open_slr_id": 17,
  "resource_page_sha256": "4cf0be3b3ffd0ac78499998adaec48909d16431a0717575beee80cf175520fbc",
  "schema_version": 1,
  "scope": "noise_only"
}

RIRS metadata:
{
  "archive_sha256": "3b50cfde915b3984738169b4beb341e9f6b8062ae4c2076146c5db71c2c05dc7",
  "corpus": "RIRS_NOISES",
  "declared_license": "Apache 2.0",
  "excluded_real_area_noise_files": 92,
  "license_sha256": "cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30",
  "open_slr_id": 28,
  "real_rirs": 325,
  "resour

## 3. Deterministic corpus audit and 50/50 inventories


In [ ]:
import json
import shutil
import sys
from pathlib import Path

REPO = Path("/content/voice-spoofing-detection-v1b")
WORK_ROOT = Path("/content/v1b_work")
DOWNLOAD_DIR = WORK_ROOT / "downloads"
EXTRACT_DIR = WORK_ROOT / "extracted"
EVIDENCE_DIR = WORK_ROOT / "evidence"

CHECKPOINT_ROOT = Path(
    "/content/drive/MyDrive/voice-spoofing-detection-v1b-checkpoint"
)
CHECKPOINT_MANIFEST_DIR = (
    CHECKPOINT_ROOT
    / "data"
    / "manifests"
    / "augmentation"
)

MUSAN_NOISE_ROOT = (
    EXTRACT_DIR / "musan" / "musan" / "noise"
)
RIRS_FILTERED_ROOT = (
    WORK_ROOT / "filtered" / "rirs_only"
)

MUSAN_ARCHIVE = DOWNLOAD_DIR / "musan.tar.gz"
RIRS_ARCHIVE = DOWNLOAD_DIR / "rirs_noises.zip"

AUDIT_OUTPUT_DIR = (
    REPO / "data" / "manifests" / "augmentation"
)

assert not AUDIT_OUTPUT_DIR.exists(), AUDIT_OUTPUT_DIR
CHECKPOINT_MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

sys.path.insert(0, str(REPO))

from src.audit.augmentation_corpora import audit_corpus

musan_metadata = json.loads(
    (
        EVIDENCE_DIR / "musan_metadata.json"
    ).read_text(encoding="utf-8")
)

rirs_metadata = json.loads(
    (
        EVIDENCE_DIR / "rirs_metadata.json"
    ).read_text(encoding="utf-8")
)

reports = {}

reports["MUSAN"] = audit_corpus(
    root=MUSAN_NOISE_ROOT,
    archive=MUSAN_ARCHIVE,
    license_file=(
        EVIDENCE_DIR / "CC-BY-4.0.txt"
    ),
    expected_archive_sha256=(
        musan_metadata["archive_sha256"]
    ),
    expected_license_sha256=(
        musan_metadata["license_sha256"]
    ),
    output_dir=AUDIT_OUTPUT_DIR,
    corpus_name="MUSAN",
    seed=2026,
)

reports["RIRS_NOISES"] = audit_corpus(
    root=RIRS_FILTERED_ROOT,
    archive=RIRS_ARCHIVE,
    license_file=(
        EVIDENCE_DIR / "Apache-2.0.txt"
    ),
    expected_archive_sha256=(
        rirs_metadata["archive_sha256"]
    ),
    expected_license_sha256=(
        rirs_metadata["license_sha256"]
    ),
    output_dir=AUDIT_OUTPUT_DIR,
    corpus_name="RIRS_NOISES",
    seed=2026,
)

for source in sorted(
    AUDIT_OUTPUT_DIR.iterdir()
):
    if source.is_file():
        shutil.copy2(
            source,
            CHECKPOINT_MANIFEST_DIR
            / source.name,
        )

print(
    json.dumps(
        reports,
        indent=2,
        sort_keys=True,
    )
)

print("\nGenerated and checkpointed files:")

for path in sorted(
    CHECKPOINT_MANIFEST_DIR.iterdir()
):
    print(
        path.name,
        path.stat().st_size,
        "bytes",
    )

{
  "MUSAN": {
    "archive_sha256": "86d1061c7e15b5c9e906777685c519701df51bfde3001e1070dcc9ffac955ee1",
    "corpus": "MUSAN",
    "files": 930,
    "half_counts": {
      "selection": 465,
      "training": 465
    },
    "inventory_file_sha256": "2ce6c6e6be1bc8d5054ddbd4b90612727e682f662947dfce3910c7a4e8b5b4d8",
    "inventory_sha256": "e3e4cf39749620f77e6ace9253e3527656f0a30b1fb7c33cd8f38245b628c1bf",
    "license_sha256": "9ba9550ad48438d0836ddab3da480b3b69ffa0aac7b7878b5a0039e7ab429411",
    "schema_version": 1,
    "split_seed": 2026
  },
  "RIRS_NOISES": {
    "archive_sha256": "3b50cfde915b3984738169b4beb341e9f6b8062ae4c2076146c5db71c2c05dc7",
    "corpus": "RIRS_NOISES",
    "files": 60325,
    "half_counts": {
      "selection": 30163,
      "training": 30162
    },
    "inventory_file_sha256": "2a5d9152f15cd5666e6b78efd49d2202ad69a67fac09d7a95174c740aafd0117",
    "inventory_sha256": "eb49a73c1b4109342f8a97f81993fc1795d0e202116fae24701a728307efdd99",
    "license_sha256": "

## 4. Pin corpora and the verified FFmpeg runtime


In [ ]:
import importlib
import json
import shutil
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/voice-spoofing-detection-v1b")
WORK_ROOT = Path("/content/v1b_work")
EVIDENCE_DIR = WORK_ROOT / "evidence"

CHECKPOINT_ROOT = Path(
    "/content/drive/MyDrive/voice-spoofing-detection-v1b-checkpoint"
)
CHECKPOINT_REPO = CHECKPOINT_ROOT / "repository"

CONFIG_PATH = REPO / "configs" / "augmentation.yaml"
TRACK_V_EVIDENCE = (
    REPO
    / "docs"
    / "audit_evidence"
    / "trackV_v1b"
)

MUSAN_ARCHIVE_SHA256 = (
    "86d1061c7e15b5c9e906777685c519701df51bfde3001e1070dcc9ffac955ee1"
)
RIRS_ARCHIVE_SHA256 = (
    "3b50cfde915b3984738169b4beb341e9f6b8062ae4c2076146c5db71c2c05dc7"
)

musan_metadata = json.loads(
    (
        EVIDENCE_DIR / "musan_metadata.json"
    ).read_text(encoding="utf-8")
)

rirs_metadata = json.loads(
    (
        EVIDENCE_DIR / "rirs_metadata.json"
    ).read_text(encoding="utf-8")
)

ffmpeg_version_result = subprocess.run(
    ["ffmpeg", "-version"],
    text=True,
    capture_output=True,
    check=True,
)

ffmpeg_encoder_result = subprocess.run(
    ["ffmpeg", "-hide_banner", "-encoders"],
    text=True,
    capture_output=True,
    check=True,
)

version_lines = (
    ffmpeg_version_result.stdout.splitlines()
)
version_line = version_lines[0]

library_lines = [
    line
    for line in version_lines
    if line.lstrip().startswith("lib")
]

encoder_lines = (
    ffmpeg_encoder_result.stdout.splitlines()
)

required_encoder_names = (
    "libmp3lame",
    "libopus",
    "pcm_mulaw",
)

required_encoder_lines = [
    line.strip()
    for line in encoder_lines
    if any(
        name in line
        for name in required_encoder_names
    )
]

for name in required_encoder_names:
    assert any(
        name in line
        for line in required_encoder_lines
    ), name

ffmpeg_runtime = {
    "schema_version": 1,
    "version_line": version_line,
    "library_lines": library_lines,
    "required_encoder_lines": (
        required_encoder_lines
    ),
}

ffmpeg_evidence_path = (
    EVIDENCE_DIR / "ffmpeg_runtime.json"
)

ffmpeg_evidence_path.write_text(
    json.dumps(
        ffmpeg_runtime,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)

version_prefix = "ffmpeg version "
assert version_line.startswith(version_prefix)

ffmpeg_version = (
    version_line[len(version_prefix):]
    .split()[0]
)

library_pins = {}

for line in library_lines:
    library_name = line.split()[0]
    library_value = (
        line[len(library_name) + 1:]
    )
    library_pins[library_name] = (
        library_value
    )

config_text = CONFIG_PATH.read_text(
    encoding="utf-8"
)

old_corpus_block = """  noise:
    name: MUSAN
    open_slr_id: 17
    archive_sha256: null  # filled by V1b before corpus use
    license_sha256: null  # filled by V1b before corpus use
  rir:
    name: RIRS_NOISES
    open_slr_id: 28
    archive_sha256: null  # filled by V1b before corpus use
    license_sha256: null  # filled by V1b before corpus use
"""

new_corpus_block = f"""  noise:
    name: MUSAN
    open_slr_id: 17
    archive_sha256: "{MUSAN_ARCHIVE_SHA256}"
    license_sha256: "{musan_metadata["license_sha256"]}"
  rir:
    name: RIRS_NOISES
    open_slr_id: 28
    archive_sha256: "{RIRS_ARCHIVE_SHA256}"
    license_sha256: "{rirs_metadata["license_sha256"]}"
"""

assert old_corpus_block in config_text
config_text = config_text.replace(
    old_corpus_block,
    new_corpus_block,
    1,
)

old_ffmpeg_block = """ffmpeg:
  executable: ffmpeg
  expected_version: null  # filled by V1b before any real codec pass
  expected_codec_libraries: {}  # filled by V1b from the same runtime
"""

library_config_lines = "\n".join(
    f"    {name}: {json.dumps(value)}"
    for name, value
    in sorted(library_pins.items())
)

new_ffmpeg_block = f"""ffmpeg:
  executable: ffmpeg
  expected_version: {json.dumps(ffmpeg_version)}
  expected_codec_libraries:
{library_config_lines}
"""

assert old_ffmpeg_block in config_text
config_text = config_text.replace(
    old_ffmpeg_block,
    new_ffmpeg_block,
    1,
)

CONFIG_PATH.write_text(
    config_text,
    encoding="utf-8",
)

assert not TRACK_V_EVIDENCE.exists()
TRACK_V_EVIDENCE.mkdir(parents=True)

for source in sorted(EVIDENCE_DIR.iterdir()):
    if (
        source.is_file()
        and source.suffix.lower()
        in {".json", ".html", ".txt"}
    ):
        shutil.copy2(
            source,
            TRACK_V_EVIDENCE / source.name,
        )

musan_audit = json.loads(
    (
        REPO
        / "data"
        / "manifests"
        / "augmentation"
        / "musan_audit.json"
    ).read_text(encoding="utf-8")
)

rirs_audit = json.loads(
    (
        REPO
        / "data"
        / "manifests"
        / "augmentation"
        / "rirs_noises_audit.json"
    ).read_text(encoding="utf-8")
)

pins_record = {
    "schema_version": 1,
    "decisions": ["D50", "D56"],
    "split_seed": 2026,
    "musan": {
        "archive_sha256": (
            MUSAN_ARCHIVE_SHA256
        ),
        "license_sha256": (
            musan_metadata["license_sha256"]
        ),
        "files": musan_audit["files"],
        "half_counts": (
            musan_audit["half_counts"]
        ),
        "inventory_file_sha256": (
            musan_audit[
                "inventory_file_sha256"
            ]
        ),
    },
    "rirs_noises": {
        "archive_sha256": (
            RIRS_ARCHIVE_SHA256
        ),
        "license_sha256": (
            rirs_metadata["license_sha256"]
        ),
        "files": rirs_audit["files"],
        "half_counts": (
            rirs_audit["half_counts"]
        ),
        "inventory_file_sha256": (
            rirs_audit[
                "inventory_file_sha256"
            ]
        ),
    },
    "ffmpeg": {
        "version": ffmpeg_version,
        "libraries": library_pins,
        "required_encoders": sorted(
            required_encoder_names
        ),
    },
}

pins_path = (
    TRACK_V_EVIDENCE
    / "corpus_and_tool_pins.json"
)

pins_path.write_text(
    json.dumps(
        pins_record,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)

repo_text = str(REPO)

if repo_text not in sys.path:
    sys.path.insert(0, repo_text)

sys.path_importer_cache.pop(
    repo_text,
    None,
)
importlib.invalidate_caches()

from src.augmentation.codec import (
    FFmpegCodecBackend,
)
from src.augmentation.config import (
    load_augmentation_config,
)

config = load_augmentation_config(
    CONFIG_PATH
)

assert config.corpus_pins_complete
assert config.ffmpeg_pin_complete

backend = FFmpegCodecBackend(
    config.ffmpeg,
    config.sample_rate,
)
backend.verify_pin()

checkpoint_config = (
    CHECKPOINT_REPO
    / "configs"
    / "augmentation.yaml"
)
checkpoint_config.parent.mkdir(
    parents=True,
    exist_ok=True,
)
shutil.copy2(
    CONFIG_PATH,
    checkpoint_config,
)

checkpoint_evidence = (
    CHECKPOINT_REPO
    / "docs"
    / "audit_evidence"
    / "trackV_v1b"
)

if checkpoint_evidence.exists():
    shutil.rmtree(checkpoint_evidence)

shutil.copytree(
    TRACK_V_EVIDENCE,
    checkpoint_evidence,
)

print("Corpus pins complete:", config.corpus_pins_complete)
print("FFmpeg pin complete:", config.ffmpeg_pin_complete)
print("FFmpeg runtime verified:", ffmpeg_version)
print("Repository checkpoint saved:", CHECKPOINT_REPO)

status = subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO,
    text=True,
    capture_output=True,
    check=True,
)

print("\nRepository changes:")
print(status.stdout)

Corpus pins complete: True
FFmpeg pin complete: True
FFmpeg runtime verified: 6.1.1-3ubuntu5
Repository checkpoint saved: /content/drive/MyDrive/voice-spoofing-detection-v1b-checkpoint/repository

Repository changes:
 M configs/augmentation.yaml
?? docs/audit_evidence/trackV_v1b/



## 5. Restore and verify the frozen FoR archive

Before running the next cell, upload `part_000` through `part_005` to `/content`. The cell rebuilds the archive, verifies its exact byte count and SHA-256, and saves a recoverable Drive checkpoint. The archive and chunks are never committed.


In [ ]:
import hashlib
import shutil
from pathlib import Path

CONTENT_ROOT = Path("/content")
CHECKPOINT_ROOT = Path(
    "/content/drive/MyDrive/voice-spoofing-detection-v1b-checkpoint"
)

FOR_ARCHIVE = (
    CONTENT_ROOT / "for_2sec_ff8c82c7.tar"
)
CHECKPOINT_ARCHIVE = (
    CHECKPOINT_ROOT / "for_2sec_ff8c82c7.tar"
)
CHECKPOINT_REVISION = (
    CHECKPOINT_ROOT / "for_2sec_REVISION.txt"
)

EXPECTED_PART_SIZES = {
    "part_000": 209_715_200,
    "part_001": 209_715_200,
    "part_002": 209_715_200,
    "part_003": 209_715_200,
    "part_004": 209_715_200,
    "part_005": 173_342_720,
}

EXPECTED_ARCHIVE_BYTES = 1_221_918_720
EXPECTED_ARCHIVE_SHA256 = (
    "9bddb83503ab442e9c7f54a1b7b9c02bc0ce32ed93f7863f28de5f59b6540b46"
)
EXPECTED_REVISION = (
    "ff8c82c79e7bbefef811941bac9775c9328e9055"
)

for name, expected_size in EXPECTED_PART_SIZES.items():
    part = CONTENT_ROOT / name
    assert part.is_file(), f"Missing: {part}"
    assert part.stat().st_size == expected_size, (
        name,
        part.stat().st_size,
        expected_size,
    )
    print(
        name,
        part.stat().st_size,
        "expected",
        expected_size,
    )

assert not FOR_ARCHIVE.exists(), FOR_ARCHIVE

with FOR_ARCHIVE.open("wb") as output_stream:
    for name in EXPECTED_PART_SIZES:
        with (
            CONTENT_ROOT / name
        ).open("rb") as input_stream:
            shutil.copyfileobj(
                input_stream,
                output_stream,
                length=1024 * 1024,
            )

assert (
    FOR_ARCHIVE.stat().st_size
    == EXPECTED_ARCHIVE_BYTES
)

digest = hashlib.sha256()

with FOR_ARCHIVE.open("rb") as stream:
    for block in iter(
        lambda: stream.read(1024 * 1024),
        b"",
    ):
        digest.update(block)

actual_sha256 = digest.hexdigest()

assert actual_sha256 == EXPECTED_ARCHIVE_SHA256

print("\nFoR archive integrity verified")
print("Bytes:", FOR_ARCHIVE.stat().st_size)
print("SHA-256:", actual_sha256)

if CHECKPOINT_ARCHIVE.exists():
    assert (
        CHECKPOINT_ARCHIVE.stat().st_size
        == EXPECTED_ARCHIVE_BYTES
    )
else:
    shutil.copy2(
        FOR_ARCHIVE,
        CHECKPOINT_ARCHIVE,
    )

CHECKPOINT_REVISION.write_text(
    EXPECTED_REVISION + "\n",
    encoding="utf-8",
)

print(
    "Persistent FoR archive saved:",
    CHECKPOINT_ARCHIVE,
)
print(
    "Persistent revision saved:",
    CHECKPOINT_REVISION,
)

part_000 209715200 expected 209715200
part_001 209715200 expected 209715200
part_002 209715200 expected 209715200
part_003 209715200 expected 209715200
part_004 209715200 expected 209715200
part_005 173342720 expected 173342720

FoR archive integrity verified
Bytes: 1221918720
SHA-256: 9bddb83503ab442e9c7f54a1b7b9c02bc0ce32ed93f7863f28de5f59b6540b46
Persistent FoR archive saved: /content/drive/MyDrive/voice-spoofing-detection-v1b-checkpoint/for_2sec_ff8c82c7.tar
Persistent revision saved: /content/drive/MyDrive/voice-spoofing-detection-v1b-checkpoint/for_2sec_REVISION.txt


## 6. Extract and hash-check frozen `val-A` only


In [ ]:
import csv
import hashlib
import shutil
import tarfile
from pathlib import Path, PurePosixPath

REPO = Path("/content/voice-spoofing-detection-v1b")
FOR_ARCHIVE = Path("/content/for_2sec_ff8c82c7.tar")
FROZEN_MANIFEST = (
    REPO
    / "data"
    / "manifests"
    / "for2sec_frozen_manifest.csv"
)

VAL_A_ROOT = Path("/content/data/for_2sec")

assert FOR_ARCHIVE.is_file()
assert FROZEN_MANIFEST.is_file()
assert not VAL_A_ROOT.exists(), VAL_A_ROOT

with FROZEN_MANIFEST.open(
    "r",
    encoding="utf-8",
    newline="",
) as stream:
    manifest_rows = list(csv.DictReader(stream))

val_a_rows = [
    row
    for row in manifest_rows
    if row["split"] == "val-A"
]

expected_by_path = {
    row["path"]: row
    for row in val_a_rows
}

assert len(expected_by_path) == 1_330

VAL_A_ROOT.mkdir(parents=True)
resolved_root = VAL_A_ROOT.resolve()
extracted_paths = set()

with tarfile.open(
    FOR_ARCHIVE,
    mode="r:",
) as archive:
    for member in archive:
        if not member.isfile():
            continue

        member_path = PurePosixPath(
            member.name
        )

        if (
            member_path.is_absolute()
            or ".." in member_path.parts
            or not member_path.parts
            or member_path.parts[0] != "for_2sec"
        ):
            continue

        relative = PurePosixPath(
            *member_path.parts[1:]
        ).as_posix()

        if relative not in expected_by_path:
            continue

        target = (
            VAL_A_ROOT / relative
        ).resolve()

        assert target.is_relative_to(
            resolved_root
        )

        target.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        source_stream = archive.extractfile(
            member
        )
        assert source_stream is not None

        with (
            source_stream,
            target.open("wb")
            as destination_stream,
        ):
            shutil.copyfileobj(
                source_stream,
                destination_stream,
                length=1024 * 1024,
            )

        extracted_paths.add(relative)

assert extracted_paths == set(
    expected_by_path
)

def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        for block in iter(
            lambda: stream.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()

mismatches = []

for relative, row in expected_by_path.items():
    actual = file_sha256(
        VAL_A_ROOT / relative
    )

    if actual != row["hash"]:
        mismatches.append(relative)

assert not mismatches, mismatches[:5]

print("Extracted val-A files:", len(extracted_paths))
print("Hash mismatches:", len(mismatches))
print("val-A root:", VAL_A_ROOT)

Extracted val-A files: 1330
Hash mismatches: 0
val-A root: /content/data/for_2sec


## 7. Selection-set preflight


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf

REPO = Path("/content/voice-spoofing-detection-v1b")
VAL_A_ROOT = Path("/content/data/for_2sec")
MUSAN_NOISE_ROOT = (
    Path("/content/v1b_work/extracted/musan/musan/noise")
)
RIRS_FILTERED_ROOT = (
    Path("/content/v1b_work/filtered/rirs_only")
)
MANIFEST_DIR = REPO / "data" / "manifests" / "augmentation"

sys.path.insert(0, str(REPO))

from src.augmentation.codec import FFmpegCodecBackend
from src.augmentation.config import load_augmentation_config

config = load_augmentation_config(
    REPO / "configs" / "augmentation.yaml"
)

noise_inventory = pd.read_csv(
    MANIFEST_DIR / "musan_inventory.csv"
)
rir_inventory = pd.read_csv(
    MANIFEST_DIR / "rirs_noises_inventory.csv"
)
for_manifest = pd.read_csv(
    REPO / "data" / "manifests" / "for2sec_frozen_manifest.csv"
)

noise_selection = (
    noise_inventory.loc[noise_inventory["half"] == "selection"]
    .sort_values("rel_path")
    .reset_index(drop=True)
)

rir_selection = (
    rir_inventory.loc[rir_inventory["half"] == "selection"]
    .sort_values("rel_path")
    .reset_index(drop=True)
)

val_a = (
    for_manifest.loc[for_manifest["split"] == "val-A"]
    .sort_values("path")
    .reset_index(drop=True)
)

assert len(noise_selection) == 465
assert len(rir_selection) == 30_163
assert len(val_a) == 1_330

print("MUSAN selection formats:")
print(
    noise_selection.groupby(
        ["samplerate", "channels"]
    ).size()
)

print("\nRIRS selection formats:")
print(
    rir_selection.groupby(
        ["samplerate", "channels"]
    ).size()
)

source_path = VAL_A_ROOT / val_a.iloc[0]["path"]
source_waveform, source_rate = sf.read(
    source_path,
    dtype="float32",
    always_2d=False,
)

assert source_rate == 16_000
assert source_waveform.ndim == 1
assert source_waveform.size == 32_000
assert np.isfinite(source_waveform).all()

backend = FFmpegCodecBackend(
    config.ffmpeg,
    config.sample_rate,
)

options = {
    option["id"]: option
    for option in config.codec_options
}

opus_output = backend.round_trip(
    source_waveform,
    options["opus_16"],
)

narrowband_output = backend.round_trip(
    source_waveform,
    options["narrowband_mulaw"],
)

assert opus_output.shape == source_waveform.shape
assert narrowband_output.shape == source_waveform.shape
assert np.isfinite(opus_output).all()
assert np.isfinite(narrowband_output).all()

print("\nPreflight source:", val_a.iloc[0]["path"])
print("Source samples:", source_waveform.size)
print("Opus 16 kbps output:", opus_output.shape)
print("Narrowband μ-law output:", narrowband_output.shape)
print("\nSelection-set preflight passed")

MUSAN selection formats:
samplerate  channels
16000       1           465
dtype: int64

RIRS selection formats:
samplerate  channels
16000       1           30019
            2              52
            8              20
            16             37
            30             35
dtype: int64

Preflight source: FAKE/file1001.wav_16k.wav_norm.wav_mono.wav_silence.wav_2sec.wav
Source samples: 32000
Opus 16 kbps output: (32000,)
Narrowband μ-law output: (32000,)

Selection-set preflight passed


## 8. Generate selection set S

For every one of the 1,330 frozen `val-A` sources, this cell records clean, recorded-noise, reverberation, Opus 16 kbps and narrowband mu-law conditions. It checkpoints every 25 sources and commits only manifests and hashes, never generated audio.


In [ ]:
import csv
import hashlib
import importlib
import json
import shutil
import subprocess
import sys
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from scipy import signal

REPO = Path("/content/voice-spoofing-detection-v1b")
VAL_A_ROOT = Path("/content/data/for_2sec")
MUSAN_NOISE_ROOT = Path(
    "/content/v1b_work/extracted/musan/musan/noise"
)
RIRS_FILTERED_ROOT = Path(
    "/content/v1b_work/filtered/rirs_only"
)

MANIFEST_DIR = (
    REPO / "data" / "manifests" / "augmentation"
)
FINAL_MANIFEST = (
    MANIFEST_DIR / "selection_set_s.csv"
)
PARTIAL_MANIFEST = (
    MANIFEST_DIR / "selection_set_s.partial.csv"
)
REPORT_PATH = (
    MANIFEST_DIR / "selection_set_s_report.json"
)

REPO_EVIDENCE_DIR = (
    REPO
    / "docs"
    / "audit_evidence"
    / "trackV_v1b"
)

CHECKPOINT_ROOT = Path(
    "/content/drive/MyDrive/voice-spoofing-detection-v1b-checkpoint"
)
CHECKPOINT_MANIFEST_DIR = (
    CHECKPOINT_ROOT
    / "data"
    / "manifests"
    / "augmentation"
)
CHECKPOINT_EVIDENCE_DIR = (
    CHECKPOINT_ROOT
    / "repository"
    / "docs"
    / "audit_evidence"
    / "trackV_v1b"
)
DRIVE_PARTIAL = (
    CHECKPOINT_MANIFEST_DIR
    / "selection_set_s.partial.csv"
)

CHECKPOINT_MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
CHECKPOINT_EVIDENCE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SELECTION_SEED = 2026
CONDITIONS = (
    "clean",
    "recorded_noise_10db",
    "reverberation",
    "opus_16",
    "narrowband_mulaw",
)

assert not FINAL_MANIFEST.exists(), FINAL_MANIFEST
assert VAL_A_ROOT.is_dir()
assert MUSAN_NOISE_ROOT.is_dir()
assert RIRS_FILTERED_ROOT.is_dir()

repo_text = str(REPO)

if repo_text not in sys.path:
    sys.path.insert(0, repo_text)

sys.path_importer_cache.pop(
    repo_text,
    None,
)
importlib.invalidate_caches()

from src.augmentation.codec import FFmpegCodecBackend
from src.augmentation.config import load_augmentation_config

config = load_augmentation_config(
    REPO / "configs" / "augmentation.yaml"
)

codec_backend = FFmpegCodecBackend(
    config.ffmpeg,
    config.sample_rate,
)
codec_backend.verify_pin()

codec_options = {
    option["id"]: option
    for option in config.codec_options
}

noise_inventory = pd.read_csv(
    MANIFEST_DIR / "musan_inventory.csv"
)
rir_inventory = pd.read_csv(
    MANIFEST_DIR / "rirs_noises_inventory.csv"
)
for_manifest = pd.read_csv(
    REPO
    / "data"
    / "manifests"
    / "for2sec_frozen_manifest.csv"
)

noise_selection = (
    noise_inventory.loc[
        noise_inventory["half"] == "selection"
    ]
    .sort_values("rel_path")
    .reset_index(drop=True)
)

rir_selection = (
    rir_inventory.loc[
        rir_inventory["half"] == "selection"
    ]
    .sort_values("rel_path")
    .reset_index(drop=True)
)

val_a = (
    for_manifest.loc[
        for_manifest["split"] == "val-A"
    ]
    .sort_values("path")
    .reset_index(drop=True)
)

assert len(noise_selection) == 465
assert len(rir_selection) == 30_163
assert len(val_a) == 1_330

FIELDNAMES = [
    "source_path",
    "label",
    "subgroup",
    "group",
    "source_file_sha256",
    "condition",
    "condition_seed",
    "resource_path",
    "resource_file_sha256",
    "resource_channel",
    "parameters_json",
    "sample_rate_hz",
    "samples",
    "waveform_float32_sha256",
]

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        for block in iter(
            lambda: stream.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()

def waveform_sha256(
    waveform: np.ndarray,
) -> str:
    canonical = np.asarray(
        waveform,
        dtype=np.float32,
    ).astype("<f4", copy=False)

    return hashlib.sha256(
        canonical.tobytes(order="C")
    ).hexdigest()

def derived_seed(
    source_path: str,
    condition: str,
) -> int:
    payload = (
        f"{SELECTION_SEED}\0"
        f"{source_path}\0"
        f"{condition}"
    ).encode("utf-8")

    digest = hashlib.sha256(payload).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="big",
        signed=False,
    )

def fit_noise(
    noise: np.ndarray,
    samples: int,
    rng: np.random.RandomState,
):
    if noise.size < samples:
        repeats = int(
            np.ceil(samples / noise.size)
        )

        return (
            np.tile(noise, repeats)[:samples],
            {
                "fit": "tile",
                "repeats": repeats,
                "start_frame": 0,
            },
        )

    if noise.size > samples:
        start = int(
            rng.randint(
                0,
                noise.size - samples + 1,
            )
        )

        return (
            noise[start:start + samples],
            {
                "fit": "crop",
                "repeats": 1,
                "start_frame": start,
            },
        )

    return (
        noise,
        {
            "fit": "exact",
            "repeats": 1,
            "start_frame": 0,
        },
    )

def apply_condition(
    source: np.ndarray,
    source_path: str,
    condition: str,
):
    condition_seed = derived_seed(
        source_path,
        condition,
    )
    rng = np.random.RandomState(
        condition_seed
    )

    resource_path = ""
    resource_sha256 = ""
    resource_channel = ""
    parameters = {}

    if condition == "clean":
        output = source.copy()

    elif condition == "recorded_noise_10db":
        resource_index = int(
            rng.randint(
                0,
                len(noise_selection),
            )
        )
        resource_row = noise_selection.iloc[
            resource_index
        ]

        resource_path = str(
            resource_row["rel_path"]
        )
        resource_sha256 = str(
            resource_row["sha256"]
        )

        noise_path = (
            MUSAN_NOISE_ROOT / resource_path
        )
        noise_data, noise_rate = sf.read(
            noise_path,
            dtype="float32",
            always_2d=True,
        )

        assert noise_rate == 16_000
        assert noise_data.shape[1] == 1

        resource_channel = 0
        noise = noise_data[:, 0]

        noise, fit_parameters = fit_noise(
            noise,
            source.size,
            rng,
        )

        noise = noise.astype(np.float64)
        noise -= float(np.mean(noise))

        source_rms = float(
            np.sqrt(
                np.mean(
                    source.astype(
                        np.float64
                    ) ** 2
                )
            )
        )
        noise_rms = float(
            np.sqrt(np.mean(noise ** 2))
        )

        assert source_rms > 0
        assert noise_rms > 0

        snr_db = 10.0
        scale = source_rms / (
            noise_rms
            * 10.0 ** (snr_db / 20.0)
        )

        output = np.asarray(
            source.astype(np.float64)
            + noise * scale,
            dtype=np.float32,
        )

        parameters = {
            "snr_db": snr_db,
            "resource_index": resource_index,
            **fit_parameters,
        }

    elif condition == "reverberation":
        resource_index = int(
            rng.randint(
                0,
                len(rir_selection),
            )
        )
        resource_row = rir_selection.iloc[
            resource_index
        ]

        resource_path = str(
            resource_row["rel_path"]
        )
        resource_sha256 = str(
            resource_row["sha256"]
        )

        rir_path = (
            RIRS_FILTERED_ROOT
            / resource_path
        )
        rir_data, rir_rate = sf.read(
            rir_path,
            dtype="float32",
            always_2d=True,
        )

        assert rir_rate == 16_000

        resource_channel = int(
            rng.randint(
                0,
                rir_data.shape[1],
            )
        )
        impulse = rir_data[
            :,
            resource_channel,
        ]

        output = signal.fftconvolve(
            source.astype(np.float64),
            impulse.astype(np.float64),
            mode="full",
        )[:source.size].astype(np.float32)

        parameters = {
            "resource_index": resource_index,
            "channel_policy": (
                "seeded_single_channel"
            ),
            "convolution": (
                "full_then_crop_to_input"
            ),
        }

    elif condition == "opus_16":
        output = codec_backend.round_trip(
            source,
            codec_options["opus_16"],
        )

        parameters = {
            "codec_option": "opus_16",
            "bitrate_kbps": 16,
            "vbr": False,
            "application": "voip",
        }

    elif condition == "narrowband_mulaw":
        output = codec_backend.round_trip(
            source,
            codec_options[
                "narrowband_mulaw"
            ],
        )

        parameters = {
            "codec_option": (
                "narrowband_mulaw"
            ),
            "intermediate_sample_rate_hz": 8000,
            "codec": "pcm_mulaw",
            "output_sample_rate_hz": 16000,
        }

    else:
        raise ValueError(condition)

    output = np.asarray(
        output,
        dtype=np.float32,
    )

    assert output.shape == source.shape
    assert np.isfinite(output).all()

    return {
        "output": output,
        "condition_seed": condition_seed,
        "resource_path": resource_path,
        "resource_file_sha256": (
            resource_sha256
        ),
        "resource_channel": (
            resource_channel
        ),
        "parameters": parameters,
    }

# Restore a saved partial manifest if one exists.
if (
    not PARTIAL_MANIFEST.exists()
    and DRIVE_PARTIAL.exists()
):
    shutil.copy2(
        DRIVE_PARTIAL,
        PARTIAL_MANIFEST,
    )
    print(
        "Restored partial manifest from Drive"
    )

completed_sources = set()

if PARTIAL_MANIFEST.exists():
    with PARTIAL_MANIFEST.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as stream:
        existing_rows = list(
            csv.DictReader(stream)
        )

    grouped_rows = defaultdict(list)

    for row in existing_rows:
        grouped_rows[
            row["source_path"]
        ].append(row)

    valid_sources = {
        source_path
        for source_path, rows
        in grouped_rows.items()
        if (
            len(rows) == len(CONDITIONS)
            and {
                row["condition"]
                for row in rows
            } == set(CONDITIONS)
        )
    }

    valid_rows = [
        row
        for row in existing_rows
        if row["source_path"]
        in valid_sources
    ]

    if len(valid_rows) != len(
        existing_rows
    ):
        with PARTIAL_MANIFEST.open(
            "w",
            encoding="utf-8",
            newline="",
        ) as stream:
            writer = csv.DictWriter(
                stream,
                fieldnames=FIELDNAMES,
                lineterminator="\n",
            )
            writer.writeheader()
            writer.writerows(valid_rows)

        print(
            "Removed incomplete partial rows"
        )

    completed_sources = set(
        valid_sources
    )

    print(
        "Resuming after",
        len(completed_sources),
        "completed source files",
    )

else:
    with PARTIAL_MANIFEST.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as stream:
        writer = csv.DictWriter(
            stream,
            fieldnames=FIELDNAMES,
            lineterminator="\n",
        )
        writer.writeheader()

for _, source_row in val_a.iterrows():
    source_relative = str(
        source_row["path"]
    )

    if source_relative in completed_sources:
        continue

    source_file = (
        VAL_A_ROOT / source_relative
    )

    source_waveform, source_rate = sf.read(
        source_file,
        dtype="float32",
        always_2d=False,
    )

    assert source_rate == 16_000
    assert source_waveform.ndim == 1
    assert source_waveform.size == 32_000
    assert np.isfinite(
        source_waveform
    ).all()

    source_file_hash = sha256_file(
        source_file
    )

    assert source_file_hash == str(
        source_row["hash"]
    )

    output_rows = []

    for condition in CONDITIONS:
        result = apply_condition(
            source_waveform,
            source_relative,
            condition,
        )

        output_rows.append({
            "source_path": source_relative,
            "label": str(
                source_row["label"]
            ),
            "subgroup": str(
                source_row["subgroup"]
            ),
            "group": int(
                source_row["group"]
            ),
            "source_file_sha256": (
                source_file_hash
            ),
            "condition": condition,
            "condition_seed": result[
                "condition_seed"
            ],
            "resource_path": result[
                "resource_path"
            ],
            "resource_file_sha256": result[
                "resource_file_sha256"
            ],
            "resource_channel": result[
                "resource_channel"
            ],
            "parameters_json": json.dumps(
                result["parameters"],
                sort_keys=True,
                separators=(",", ":"),
            ),
            "sample_rate_hz": 16_000,
            "samples": 32_000,
            "waveform_float32_sha256": (
                waveform_sha256(
                    result["output"]
                )
            ),
        })

    with PARTIAL_MANIFEST.open(
        "a",
        encoding="utf-8",
        newline="",
    ) as stream:
        writer = csv.DictWriter(
            stream,
            fieldnames=FIELDNAMES,
            lineterminator="\n",
        )
        writer.writerows(output_rows)

    completed_sources.add(
        source_relative
    )

    if (
        len(completed_sources) % 25 == 0
        or len(completed_sources)
        == len(val_a)
    ):
        shutil.copy2(
            PARTIAL_MANIFEST,
            DRIVE_PARTIAL,
        )

        print(
            f"Completed "
            f"{len(completed_sources)}"
            f"/{len(val_a)} source files; "
            f"checkpoint saved"
        )

with PARTIAL_MANIFEST.open(
    "r",
    encoding="utf-8",
    newline="",
) as stream:
    final_rows = list(
        csv.DictReader(stream)
    )

assert len(final_rows) == 6_650

condition_counts = Counter(
    row["condition"]
    for row in final_rows
)

assert dict(condition_counts) == {
    condition: 1_330
    for condition in CONDITIONS
}

PARTIAL_MANIFEST.replace(
    FINAL_MANIFEST
)

semantic_payload = json.dumps(
    final_rows,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)

semantic_sha256 = hashlib.sha256(
    semantic_payload.encode("utf-8")
).hexdigest()

musan_audit = json.loads(
    (
        MANIFEST_DIR / "musan_audit.json"
    ).read_text(encoding="utf-8")
)

rirs_audit = json.loads(
    (
        MANIFEST_DIR
        / "rirs_noises_audit.json"
    ).read_text(encoding="utf-8")
)

repo_base_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    text=True,
    capture_output=True,
    check=True,
).stdout.strip()

report = {
    "schema_version": 1,
    "name": (
        "D50 selection condition set S"
    ),
    "decisions": [
        "D49",
        "D50",
        "D56",
    ],
    "selection_seed": SELECTION_SEED,
    "repo_base_commit": repo_base_commit,
    "for_revision": (
        "ff8c82c79e7bbefef811941bac9775c9328e9055"
    ),
    "for_archive_sha256": (
        "9bddb83503ab442e9c7f54a1b7b9c02bc0ce32ed93f7863f28de5f59b6540b46"
    ),
    "source_manifest_sha256": (
        sha256_file(
            REPO
            / "data"
            / "manifests"
            / "for2sec_frozen_manifest.csv"
        )
    ),
    "source_split": "val-A",
    "source_files": 1_330,
    "conditions": list(CONDITIONS),
    "condition_counts": dict(
        sorted(condition_counts.items())
    ),
    "records": len(final_rows),
    "manifest_file": FINAL_MANIFEST.name,
    "manifest_file_sha256": (
        sha256_file(FINAL_MANIFEST)
    ),
    "manifest_semantic_sha256": (
        semantic_sha256
    ),
    "waveform_hash_encoding": (
        "little-endian float32 sample bytes"
    ),
    "resource_choice_policy": (
        "SHA-256-derived per-source/"
        "per-condition RandomState seed "
        "from global seed 2026"
    ),
    "rir_channel_policy": (
        "one seeded channel from the "
        "selected RIR file; channel "
        "index recorded"
    ),
    "audio_outputs_committed": False,
    "musan_inventory_sha256": (
        musan_audit[
            "inventory_file_sha256"
        ]
    ),
    "rirs_inventory_sha256": (
        rirs_audit[
            "inventory_file_sha256"
        ]
    ),
    "ffmpeg_version": "6.1.1-3ubuntu5",
}

REPORT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)

shutil.copy2(
    FINAL_MANIFEST,
    CHECKPOINT_MANIFEST_DIR
    / FINAL_MANIFEST.name,
)
shutil.copy2(
    REPORT_PATH,
    CHECKPOINT_MANIFEST_DIR
    / REPORT_PATH.name,
)
shutil.copy2(
    REPORT_PATH,
    REPO_EVIDENCE_DIR
    / REPORT_PATH.name,
)
shutil.copy2(
    REPORT_PATH,
    CHECKPOINT_EVIDENCE_DIR
    / REPORT_PATH.name,
)

pins_path = (
    REPO_EVIDENCE_DIR
    / "corpus_and_tool_pins.json"
)

pins_record = json.loads(
    pins_path.read_text(encoding="utf-8")
)

pins_record["selection_set_s"] = {
    "manifest_file_sha256": report[
        "manifest_file_sha256"
    ],
    "manifest_semantic_sha256": report[
        "manifest_semantic_sha256"
    ],
    "records": report["records"],
    "source_files": report[
        "source_files"
    ],
    "conditions": report["conditions"],
}

pins_path.write_text(
    json.dumps(
        pins_record,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)

shutil.copy2(
    pins_path,
    CHECKPOINT_EVIDENCE_DIR
    / pins_path.name,
)

print("\nSelection set S complete")
print(
    json.dumps(
        report,
        indent=2,
        sort_keys=True,
    )
)

Completed 25/1330 source files; checkpoint saved
Completed 50/1330 source files; checkpoint saved
Completed 75/1330 source files; checkpoint saved
Completed 100/1330 source files; checkpoint saved
Completed 125/1330 source files; checkpoint saved
Completed 150/1330 source files; checkpoint saved
Completed 175/1330 source files; checkpoint saved
Completed 200/1330 source files; checkpoint saved
Completed 225/1330 source files; checkpoint saved
Completed 250/1330 source files; checkpoint saved
Completed 275/1330 source files; checkpoint saved
Completed 300/1330 source files; checkpoint saved
Completed 325/1330 source files; checkpoint saved
Completed 350/1330 source files; checkpoint saved
Completed 375/1330 source files; checkpoint saved
Completed 400/1330 source files; checkpoint saved
Completed 425/1330 source files; checkpoint saved
Completed 450/1330 source files; checkpoint saved
Completed 475/1330 source files; checkpoint saved
Completed 500/1330 source files; checkpoint saved
Com

## 9. Structural validation of the persisted result


In [4]:
import csv
import hashlib
import json
import re
import shutil
from collections import Counter
from pathlib import Path

REPO = Path("/content/voice-spoofing-detection-v1b")
MANIFEST_DIR = (
    REPO / "data" / "manifests" / "augmentation"
)
EVIDENCE_DIR = (
    REPO
    / "docs"
    / "audit_evidence"
    / "trackV_v1b"
)

CHECKPOINT_ROOT = Path(
    "/content/drive/MyDrive/voice-spoofing-detection-v1b-checkpoint"
)
CHECKPOINT_EVIDENCE = (
    CHECKPOINT_ROOT
    / "repository"
    / "docs"
    / "audit_evidence"
    / "trackV_v1b"
)

SELECTION_MANIFEST = (
    MANIFEST_DIR / "selection_set_s.csv"
)
SELECTION_REPORT = (
    MANIFEST_DIR / "selection_set_s_report.json"
)
FROZEN_MANIFEST = (
    REPO
    / "data"
    / "manifests"
    / "for2sec_frozen_manifest.csv"
)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        for block in iter(
            lambda: stream.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()

def derived_seed(
    source_path: str,
    condition: str,
) -> int:
    payload = (
        f"2026\0{source_path}\0{condition}"
    ).encode("utf-8")

    digest = hashlib.sha256(
        payload
    ).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="big",
        signed=False,
    )

def read_csv(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as stream:
        return list(csv.DictReader(stream))

selection_rows = read_csv(
    SELECTION_MANIFEST
)
frozen_rows = read_csv(
    FROZEN_MANIFEST
)
noise_rows = read_csv(
    MANIFEST_DIR / "musan_inventory.csv"
)
rir_rows = read_csv(
    MANIFEST_DIR
    / "rirs_noises_inventory.csv"
)

report = json.loads(
    SELECTION_REPORT.read_text(
        encoding="utf-8"
    )
)

val_a_by_path = {
    row["path"]: row
    for row in frozen_rows
    if row["split"] == "val-A"
}

noise_selection = sorted(
    (
        row
        for row in noise_rows
        if row["half"] == "selection"
    ),
    key=lambda row: row["rel_path"],
)

rir_selection = sorted(
    (
        row
        for row in rir_rows
        if row["half"] == "selection"
    ),
    key=lambda row: row["rel_path"],
)

noise_by_path = {
    row["rel_path"]: row
    for row in noise_selection
}
rir_by_path = {
    row["rel_path"]: row
    for row in rir_selection
}

assert len(val_a_by_path) == 1_330
assert len(noise_selection) == 465
assert len(rir_selection) == 30_163
assert len(selection_rows) == 6_650

conditions = {
    "clean",
    "recorded_noise_10db",
    "reverberation",
    "opus_16",
    "narrowband_mulaw",
}

hex_pattern = re.compile(
    r"^[0-9a-f]{64}$"
)

seen_keys = set()
condition_counts = Counter()
source_counts = Counter()
resource_records_checked = 0

for row in selection_rows:
    key = (
        row["source_path"],
        row["condition"],
    )

    assert key not in seen_keys, key
    seen_keys.add(key)

    assert row["condition"] in conditions
    condition_counts[row["condition"]] += 1
    source_counts[row["source_path"]] += 1

    source = val_a_by_path[
        row["source_path"]
    ]

    assert row["label"] == source["label"]
    assert row["subgroup"] == (
        source["subgroup"]
    )
    assert row["group"] == source["group"]
    assert row["source_file_sha256"] == (
        source["hash"]
    )

    assert hex_pattern.fullmatch(
        row["source_file_sha256"]
    )
    assert hex_pattern.fullmatch(
        row["waveform_float32_sha256"]
    )

    assert int(row["condition_seed"]) == (
        derived_seed(
            row["source_path"],
            row["condition"],
        )
    )

    assert int(row["sample_rate_hz"]) == (
        16_000
    )
    assert int(row["samples"]) == 32_000

    parameters = json.loads(
        row["parameters_json"]
    )

    if row["condition"] == (
        "recorded_noise_10db"
    ):
        resource = noise_by_path[
            row["resource_path"]
        ]

        assert (
            row["resource_file_sha256"]
            == resource["sha256"]
        )
        assert int(
            row["resource_channel"]
        ) == 0
        assert float(
            parameters["snr_db"]
        ) == 10.0

        resource_index = int(
            parameters["resource_index"]
        )

        assert (
            noise_selection[
                resource_index
            ]["rel_path"]
            == row["resource_path"]
        )

        resource_records_checked += 1

    elif row["condition"] == (
        "reverberation"
    ):
        resource = rir_by_path[
            row["resource_path"]
        ]

        assert (
            row["resource_file_sha256"]
            == resource["sha256"]
        )

        channel = int(
            row["resource_channel"]
        )

        assert 0 <= channel < int(
            resource["channels"]
        )

        resource_index = int(
            parameters["resource_index"]
        )

        assert (
            rir_selection[
                resource_index
            ]["rel_path"]
            == row["resource_path"]
        )

        assert parameters[
            "convolution"
        ] == "full_then_crop_to_input"

        resource_records_checked += 1

    else:
        assert row["resource_path"] == ""
        assert (
            row["resource_file_sha256"]
            == ""
        )
        assert row["resource_channel"] == ""

        if row["condition"] == "clean":
            assert parameters == {}

        elif row["condition"] == "opus_16":
            assert parameters[
                "codec_option"
            ] == "opus_16"
            assert int(
                parameters["bitrate_kbps"]
            ) == 16

        elif row["condition"] == (
            "narrowband_mulaw"
        ):
            assert parameters[
                "codec_option"
            ] == "narrowband_mulaw"
            assert int(
                parameters[
                    "intermediate_sample_rate_hz"
                ]
            ) == 8000

assert len(seen_keys) == 6_650
assert len(source_counts) == 1_330
assert set(source_counts.values()) == {5}

assert condition_counts == {
    condition: 1_330
    for condition in conditions
}

assert resource_records_checked == 2_660

semantic_payload = json.dumps(
    selection_rows,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)

semantic_sha256 = hashlib.sha256(
    semantic_payload.encode("utf-8")
).hexdigest()

assert sha256_file(SELECTION_MANIFEST) == (
    report["manifest_file_sha256"]
)
assert semantic_sha256 == (
    report["manifest_semantic_sha256"]
)
assert sha256_file(FROZEN_MANIFEST) == (
    report["source_manifest_sha256"]
)

musan_audit = json.loads(
    (
        MANIFEST_DIR / "musan_audit.json"
    ).read_text(encoding="utf-8")
)

rirs_audit = json.loads(
    (
        MANIFEST_DIR
        / "rirs_noises_audit.json"
    ).read_text(encoding="utf-8")
)

assert sha256_file(
    MANIFEST_DIR / "musan_inventory.csv"
) == musan_audit[
    "inventory_file_sha256"
]

assert sha256_file(
    MANIFEST_DIR
    / "rirs_noises_inventory.csv"
) == rirs_audit[
    "inventory_file_sha256"
]

validation = {
    "schema_version": 1,
    "selection_manifest_sha256": (
        sha256_file(
            SELECTION_MANIFEST
        )
    ),
    "selection_semantic_sha256": (
        semantic_sha256
    ),
    "records_checked": 6_650,
    "sources_checked": 1_330,
    "resource_records_checked": 2_660,
    "unique_source_condition_keys": (
        len(seen_keys)
    ),
    "condition_counts": dict(
        sorted(condition_counts.items())
    ),
    "source_manifest_links_checked": (
        6_650
    ),
    "condition_seeds_checked": 6_650,
    "result": "passed",
    "note": (
        "Structural validation from the "
        "persisted final manifest; generation "
        "itself asserted every waveform shape "
        "and finite-value contract."
    ),
}

validation_path = (
    EVIDENCE_DIR
    / "selection_set_s_validation.json"
)

validation_path.write_text(
    json.dumps(
        validation,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)

CHECKPOINT_EVIDENCE.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copy2(
    validation_path,
    CHECKPOINT_EVIDENCE
    / validation_path.name,
)

print(
    json.dumps(
        validation,
        indent=2,
        sort_keys=True,
    )
)

{
  "condition_counts": {
    "clean": 1330,
    "narrowband_mulaw": 1330,
    "opus_16": 1330,
    "recorded_noise_10db": 1330,
    "reverberation": 1330
  },
  "condition_seeds_checked": 6650,
  "note": "Structural validation from the persisted final manifest; generation itself asserted every waveform shape and finite-value contract.",
  "records_checked": 6650,
  "resource_records_checked": 2660,
  "result": "passed",
  "schema_version": 1,
  "selection_manifest_sha256": "71a340f009b5b0a20e295f4031c15fe2471b1174444b4b0fa2d3014f34c72533",
  "selection_semantic_sha256": "8cf4858c51f78a1fd5bce78042f7d490529eb1230a149fe52a0a1f856e9046a7",
  "source_manifest_links_checked": 6650,
  "sources_checked": 1330,
  "unique_source_condition_keys": 6650
}


## Recorded result

The completed run produced 6,650 unique `(source_path, condition)` records covering 1,330 sources and five conditions. Structural validation checked all source-manifest links, deterministic seeds, resource selections, inventory hashes and manifest hashes. The authoritative values are stored in the committed JSON evidence and manifests.
